   # Derived vs catalogued amplitude (AAVSO light curves)
   Question: do amplitude, mean magnitude, SD and observation count that we compute ourselves from AAVSO observations agree with catalogued values, and why might they differ?

## Download log
- Source: AAVSO International Database (AID), photometry search
- Target: del Cep (AUID 000-BCQ-471); band: Johnson V; dates 2024-01-01 to 2025-12-31
- Downloaded 2026-10-13 (file observations_20261013_101102.csv, 480 records), no observer or campaign filter

In [1]:
import numpy as np
import pandas as pd

a = pd.read_csv("../data/raw/observations_20261013_101102.csv")
print(a.shape)
print()
print(a["observer"].value_counts())
print()
print(a["type"].value_counts())
print()
print(a["transformed"].value_counts())
print(a["fainterthan"].value_counts())
print()
print(a["mag"].describe().round(3))
print(a["uncertainty"].isna().sum(), "rows with no uncertainty")
print()
print(a.sort_values("mag").tail(4)[["jd", "mag", "observer", "transformed"]])

(480, 19)

observer
VOL     180
KPRB    153
SAH      93
WGEC     24
PJDB     22
PGC       4
BVE       2
HSHC      2
Name: count, dtype: int64

type
CCD     292
DSLR    186
PEP       2
Name: count, dtype: int64

transformed
True     293
False    187
Name: count, dtype: int64
fainterthan
False    480
Name: count, dtype: int64

count    480.000
mean       3.898
std        0.316
min        3.333
25%        3.639
50%        3.920
75%        4.113
max        6.365
Name: mag, dtype: float64
1 rows with no uncertainty

               jd    mag observer  transformed
266  2.460839e+06  4.443     KPRB         True
37   2.460442e+06  4.477     KPRB         True
311  2.460886e+06  6.283     KPRB        False
312  2.460886e+06  6.365     KPRB         True


In [2]:
def grab(key):
    return pd.to_numeric(a["comments"].str.extract(rf"\|{key}=([-\d.]+)")[0], errors="coerce")

a["check_diff"] = grab("KMAGSTD") - grab("KREFMAG")
print(a["check_diff"].notna().sum(), "rows have a usable check-star comparison")
print(a["check_diff"].describe().round(3))
print()
print(a.loc[a["check_diff"].abs() > 0.1, ["#", "jd", "mag", "observer", "check_diff"]].round(3))

def summarize(d):
    m = d["mag"]
    return {"n": len(d), "mean": m.mean(), "sd": m.std(),
            "min (brightest)": m.min(), "max (faintest)": m.max(),
            "peak-to-peak": m.max() - m.min(),
            "robust (95th-5th pct)": m.quantile(0.95) - m.quantile(0.05)}

rows = {"all 480 rows": summarize(a)}
for cut in [0.5, 0.3, 0.2]:
    keep = a["check_diff"].isna() | (a["check_diff"].abs() <= cut)
    rows[f"drop |check diff| > {cut}"] = summarize(a[keep])

print()
print(pd.DataFrame(rows).T.round(3))

199 rows have a usable check-star comparison
count    199.000
mean       0.040
std        0.254
min       -0.177
25%       -0.017
50%        0.003
75%        0.034
max        2.373
Name: check_diff, dtype: float64

       #           jd    mag observer  check_diff
37    38  2460441.841  4.477     KPRB       0.250
63    64  2460494.720  4.170     KPRB       0.129
67    68  2460500.827  4.261     KPRB       0.149
71    72  2460505.803  4.148     KPRB       0.115
76    77  2460510.790  4.174     KPRB       0.101
86    87  2460524.607  3.825     KPRB       0.181
89    90  2460530.612  3.875     KPRB       0.131
95    96  2460536.572  4.133     KPRB       0.164
175  176  2460665.607  4.028     PJDB       0.129
190  191  2460671.611  4.140     PJDB      -0.177
195  196  2460676.588  3.988     PJDB      -0.122
198  199  2460679.450  3.635     KPRB       0.145
206  207  2460688.567  4.131     PJDB      -0.108
214  215  2460695.554  3.434     PJDB      -0.101
215  216  2460697.606  3.899     PJ